Name: Lin Qizhou

Key insights / takeaways:
- I built a supervised classification pipeline to predict `default.payment.next.month` using a 25% held-out test split for evaluation.
- I compared two traditional machine learning models (e.g., Random Forest and XGBoost) against TabPFN on the same prediction task and reported key metrics (Accuracy and ROC-AUC).
- I observed how TabPFN can be competitive with standard ML baselines, and I summarized trade-offs in performance, compute cost, and setup complexity.

In [ ]:
import kagglehub
import pandas as pd
import os

dataset_path = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset")

# List contents of the downloaded dataset directory to find the data file
files_in_dataset = os.listdir(dataset_path)
print(f"Files in the dataset directory: {files_in_dataset}")

# Assuming the main data file is a CSV and is directly in the downloaded path
# You might need to adjust the filename if it's different or in a subdirectory
# For this dataset, it's typically 'UCI_Credit_Card.csv'

# Construct the full path to the CSV file
csv_file_name = 'UCI_Credit_Card.csv'
full_csv_path = os.path.join(dataset_path, csv_file_name)

# Load the CSV into a pandas DataFrame
df = pd.read_csv(full_csv_path)

print("Data loaded into DataFrame 'df' successfully.")
print(df.head())

100%|██████████| 0.98M/0.98M [00:00<00:00, 1.44MB/s]

Extracting files...
Files in the dataset directory: ['UCI_Credit_Card.csv']
Data loaded into DataFrame 'df' successfully.
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1    20000.0    2          2         1   24      2      2     -1     -1   
1   2   120000.0    2          2         2   26     -1      2      0      0   
2   3    90000.0    2          2         2   34      0      0      0      0   
3   4    50000.0    2          2         1   37      0      0      0      0   
4   5    50000.0    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...        0.0        0.0        0.0       0.0     689.0       0.0   
1  ...     3272.0     3455.0     3261.0       0.0    1000.0    1000.0   
2  ...    14331.0    14948.0    15549.0    1518.0    1500.0    1000.0   
3  ...    28314.0    28959.0    29547.0    2000.0    2019.0    1200.0   
4  ...    20940.0    19146.0    19131.

In [ ]:
df

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000.0,1,3,1,39,0,0,0,0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,29997,150000.0,1,3,2,43,-1,-1,-1,-1,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,29998,30000.0,1,2,2,37,4,3,2,-1,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,29999,80000.0,1,3,1,41,1,-1,0,0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


In [ ]:
from sklearn.model_selection import train_test_split

# Prepare features (X) and target (y)
# Dropping 'ID' as it's an identifier and the target column
X = df.drop(columns=['ID', 'default.payment.next.month'])
y = df['default.payment.next.month']

# Split the data into training and testing sets (75% train, 25% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Verify the split
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (22500, 23)
X_test shape: (7500, 23)
y_train shape: (22500,)
y_test shape: (7500,)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Initialize the models
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')

# Train the models
print("Training Random Forest...")
rf_model.fit(X_train, y_train)

print("Training XGBoost...")
xgb_model.fit(X_train, y_train)

# Generate predictions (class and probability)
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

# Calculate metrics
rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)

xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_proba)

# Print the results
print(f"Random Forest - Accuracy: {rf_acc:.4f}, ROC AUC: {rf_auc:.4f}")
print(f"XGBoost       - Accuracy: {xgb_acc:.4f}, ROC AUC: {xgb_auc:.4f}")

Training Random Forest...
Training XGBoost...
Random Forest - Accuracy: 0.8160, ROC AUC: 0.7534
XGBoost       - Accuracy: 0.8127, ROC AUC: 0.7595


In [ ]:
# Create a subset of the training data (first 2,000 samples)
X_train_sub = X_train.iloc[:2000]
y_train_sub = y_train.iloc[:2000]

print(f"TabPFN training subset shape: {X_train_sub.shape}")

# Initialize the TabPFN Classifier
tabpfn_model = TabPFNClassifier(random_state=42)

# Train the model on the subset
print("Training TabPFN...")
tabpfn_model.fit(X_train_sub, y_train_sub)

# Generate predictions on the full test set
tabpfn_pred = tabpfn_model.predict(X_test)
tabpfn_proba = tabpfn_model.predict_proba(X_test)[:, 1]

# Calculate metrics
tabpfn_acc = accuracy_score(y_test, tabpfn_pred)
tabpfn_auc = roc_auc_score(y_test, tabpfn_proba)

# Print the results
print(f"TabPFN        - Accuracy: {tabpfn_acc:.4f}, ROC AUC: {tabpfn_auc:.4f}")

TabPFN training subset shape: (2000, 23)
Training TabPFN...


Processing: 100%|██████████| [00:12<00:00]
Processing: 100%|██████████| [00:02<00:00]

TabPFN        - Accuracy: 0.8205, ROC AUC: 0.7657


In [ ]:
# Create a subset of the training data (first 2,000 samples)
X_train_sub = X_train.iloc[:2000]
y_train_sub = y_train.iloc[:2000]

print(f"TabPFN training subset shape: {X_train_sub.shape}")

# Initialize the TabPFN Classifier
tabpfn_model = TabPFNClassifier(random_state=42)

# Train the model on the subset
print("Training TabPFN...")
tabpfn_model.fit(X_train_sub, y_train_sub)

# Generate predictions on the full test set
tabpfn_pred = tabpfn_model.predict(X_test)
tabpfn_proba = tabpfn_model.predict_proba(X_test)[:, 1]

# Calculate metrics
tabpfn_acc = accuracy_score(y_test, tabpfn_pred)
tabpfn_auc = roc_auc_score(y_test, tabpfn_proba)

# Print the results
print(f"TabPFN        - Accuracy: {tabpfn_acc:.4f}, ROC AUC: {tabpfn_auc:.4f}")

TabPFN training subset shape: (2000, 23)
Training TabPFN...


Processing: 100%|██████████| [00:02<00:00]
Processing: 100%|██████████| [00:02<00:00]

TabPFN        - Accuracy: 0.8205, ROC AUC: 0.7657


In [ ]:
# Create a subset of the training data (first 2,000 samples)
X_train_sub = X_train.iloc[:2000]
y_train_sub = y_train.iloc[:2000]

print(f"TabPFN training subset shape: {X_train_sub.shape}")

# Initialize the TabPFN Classifier
tabpfn_model = TabPFNClassifier(random_state=42)

# Train the model on the subset
print("Training TabPFN...")
tabpfn_model.fit(X_train_sub, y_train_sub)

# Generate predictions on the full test set
tabpfn_pred = tabpfn_model.predict(X_test)
tabpfn_proba = tabpfn_model.predict_proba(X_test)[:, 1]

# Calculate metrics
tabpfn_acc = accuracy_score(y_test, tabpfn_pred)
tabpfn_auc = roc_auc_score(y_test, tabpfn_proba)

# Print the results
print(f"TabPFN        - Accuracy: {tabpfn_acc:.4f}, ROC AUC: {tabpfn_auc:.4f}")

TabPFN training subset shape: (2000, 23)
Training TabPFN...


Processing: 100%|██████████| [00:02<00:00]
Processing: 100%|██████████| [00:02<00:00]

TabPFN        - Accuracy: 0.8205, ROC AUC: 0.7657


In [ ]:
import sys

# Create a subset of the training data (first 2,000 samples)
X_train_sub = X_train.iloc[:2000]
y_train_sub = y_train.iloc[:2000]

print(f"TabPFN training subset shape: {X_train_sub.shape}")

# Initialize the TabPFN Classifier
tabpfn_model = TabPFNClassifier(random_state=42)

print("Training TabPFN...")

# Suppress stderr to prevent progress bars from being flagged as errors
original_stderr = sys.stderr
null_out = open(os.devnull, 'w')
sys.stderr = null_out

try:
    # Train the model on the subset
    tabpfn_model.fit(X_train_sub, y_train_sub)

    # Generate predictions on the full test set
    tabpfn_pred = tabpfn_model.predict(X_test)
    tabpfn_proba = tabpfn_model.predict_proba(X_test)[:, 1]
finally:
    # Restore stderr
    sys.stderr = original_stderr
    null_out.close()

# Calculate metrics
tabpfn_acc = accuracy_score(y_test, tabpfn_pred)
tabpfn_auc = roc_auc_score(y_test, tabpfn_proba)

# Print the results
print(f"TabPFN        - Accuracy: {tabpfn_acc:.4f}, ROC AUC: {tabpfn_auc:.4f}")

TabPFN training subset shape: (2000, 23)
Training TabPFN...
TabPFN        - Accuracy: 0.8205, ROC AUC: 0.7657


In [ ]:
import pandas as pd

# Create a dictionary with the results
results_data = {
    'Model': ['Random Forest', 'XGBoost', 'TabPFN'],
    'Accuracy': [rf_acc, xgb_acc, tabpfn_acc],
    'ROC AUC': [rf_auc, xgb_auc, tabpfn_auc]
}

# Create a DataFrame
results_df = pd.DataFrame(results_data)

# Display the DataFrame
print(results_df)

           Model  Accuracy   ROC AUC
0  Random Forest  0.816000  0.753386
1        XGBoost  0.812667  0.759529
2         TabPFN  0.820533  0.765719
